# MovieLens-100K Novelty Experiments (PopDebias + ColdBridge)

This notebook runs the novelty pipeline described in `kaggle/novelty` on MovieLens-100K using GPU-friendly BGE-M3 embeddings.

It will generate:
- `results/movielens_100k/full_results.csv`
- `results/movielens_100k/best_model.txt`
- `results/figures/*.png`

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/faroukq1/LLM-Sequential-Recommendation.git"
REPO_DIR = Path("/kaggle/working/LLM-Sequential-Recommendation")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
subprocess.run(["git", "remote", "set-url", "origin", REPO_URL], check=True)
subprocess.run(["git", "fetch", "origin"], check=True)
current_branch = subprocess.check_output(["git", "rev-parse", "--abbrev-ref", "HEAD"], text=True).strip()
subprocess.run(["git", "pull", "--ff-only", "origin", current_branch], check=True)

print("Repo:", REPO_DIR)
print("Branch:", current_branch)

In [ ]:
import importlib.util
import subprocess
import sys

def pip_install(*packages):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)

pip_install("--upgrade", "pip")
pip_install("pandas", "numpy", "scipy", "scikit-learn", "matplotlib", "tqdm", "multiprocess", "numba")

if importlib.util.find_spec("tensorflow") is None:
    pip_install("tensorflow==2.13.0")

if importlib.util.find_spec("FlagEmbedding") is None:
    pip_install("FlagEmbedding")

print("Dependencies installed.")

In [ ]:
import os
import subprocess
import sys

REPO_DIR = "/kaggle/working/LLM-Sequential-Recommendation"
WORK_DIR = "/kaggle/working/llmseqrec_novelty_ml100k"
SCRIPT_PATH = f"{REPO_DIR}/kaggle/run_novelty_ml100k.py"

# Toggle this if you want the optional BERT4Rec backbone comparison.
INCLUDE_BERT_BACKBONE = False

cmd = [
    sys.executable,
    SCRIPT_PATH,
    "--work-dir", WORK_DIR,
    "--num-epochs", "6",
    "--cores", "2",
    "--top-k", "20",
    "--candidate-k", "300",
    "--alpha-values", "0.1,0.3,0.5,0.7",
    "--tau-values", "2,3,5,10,15",
    "--decay-values", "0.5,0.7,0.8,0.9,1.0",
    "--bge-batch-size", "256",
]

if INCLUDE_BERT_BACKBONE:
    cmd.append("--include-bert-backbone")

env = os.environ.copy()
if env.get("PYTHONPATH"):
    env["PYTHONPATH"] = f"{REPO_DIR}:{env['PYTHONPATH']}"
else:
    env["PYTHONPATH"] = REPO_DIR

print("Running command:")
print(" ".join(cmd))
subprocess.run(cmd, check=True, cwd=REPO_DIR, env=env)
print("Done. Artifacts written to:", WORK_DIR)

In [ ]:
from pathlib import Path
import pandas as pd

work_dir = Path("/kaggle/working/llmseqrec_novelty_ml100k")
results_csv = work_dir / "results" / "movielens_100k" / "full_results.csv"
best_txt = work_dir / "results" / "movielens_100k" / "best_model.txt"
figures_dir = work_dir / "results" / "figures"

if results_csv.exists():
    df = pd.read_csv(results_csv)
    display(df.head(20))
else:
    print("Results file not found:", results_csv)

if best_txt.exists():
    print("=== best_model.txt ===")
    print(best_txt.read_text(encoding="utf-8"))
else:
    print("Best-model file not found:", best_txt)

if figures_dir.exists():
    print("Figures:")
    for p in sorted(figures_dir.glob("*.png")):
        print("-", p.name)

If Kaggle time is tight, reduce training cost first by setting `--num-epochs 3` and `INCLUDE_BERT_BACKBONE = False`.